# Практика: vLLM serve и OpenAI-compatible API

> Этот раздел — быстрое практическое введение. Оптимизации инференса (PagedAttention, continuous batching, speculative decoding) разберём в следующей лекции.

## vllm serve: поднимаем VLM-модель

**vLLM** — высокопроизводительный inference-сервер с OpenAI-compatible API. Устанавливается через pip:

```bash
pip install vllm
```

Для запуска любой VLM-модели с HuggingFace достаточно одной команды. Пример с **Qwen3-VL-8B-Instruct-FP8**:

```bash
vllm serve Qwen/Qwen3-VL-8B-Instruct-FP8 \
    --max-model-len 16384 \
    --max-num-seqs 16 \
    --gpu-memory-utilization 0.85 \
    --served-model-name qwen3-vl \
    --limit-mm-per-prompt '{"image": 8, "video": 1}' \
    --port 8000
```

После старта сервер пишет в лог:
```
INFO:     Started server process [...]
INFO:     Uvicorn running on http://0.0.0.0:8000
```

и слушает запросы на `/v1/chat/completions`, `/v1/models` и других OpenAI-совместимых эндпоинтах.

### Основные параметры

| Параметр | Тип / По умолчанию | Описание |
|----------|-------------------|----------|
| `--max-model-len` | int / auto | Максимальная длина контекста (prompt + output) в токенах. Для VLM особенно важен: изображение 1024×1024 при tile-size 448 даёт ~3 000+ визуальных токенов. Уменьшение снижает потребление VRAM под KV-кеш. Поддерживает суффиксы: `16k`, `32K` |
| `--max-num-seqs` | int / 256 | Максимальное число одновременно обрабатываемых запросов (sequences) за один шаг планировщика. Уменьшить, если OOM при высоком параллелизме |
| `--gpu-memory-utilization` | float / 0.92 | Доля VRAM, выделяемая vLLM (модель + KV-кеш + буферы). 0.92 — дефолт. Для VLM со многими изображениями разумно снизить до 0.80–0.85, чтобы оставить место под multimodal preprocessor |
| `--served-model-name` | str / равно `--model` | Имя модели, которое вернёт `/v1/models` и которое нужно указывать в поле `model` запросов. Полезно, если полный HF-путь неудобен |
| `--limit-mm-per-prompt` | JSON / `{}` | Ограничение на количество медиа-объектов в одном запросе. `'{"image": 8}'` — не более 8 картинок на сообщение. Защищает от случайных OOM при больших батчах изображений |
| `--tensor-parallel-size` | int / 1 | Число GPU для tensor parallelism. Для 8B-модели на одной A100 не нужен; для 72B+ ставить кратно числу карт |
| `--dtype` | auto/bf16/fp16 | Тип данных весов. FP8-чекпоинты (как наш пример) загружаются через quantization, dtype обычно выставлять не нужно |
| `--trust-remote-code` | bool / False | Разрешить исполнение кода из репозитория модели. Нужно для некоторых моделей (Phi-3.5-Vision и др.) |
| `--mm-processor-kwargs` | JSON | Дополнительные аргументы для multimodal preprocessor. Например, `'{"do_pan_and_scan": true}'` для Gemma 3 |

**Типичная ошибка при запуске VLM:**
```
ValueError: The model's max seq len (32768) is larger than the maximum
number of tokens that can be stored in KV cache (12288). Try increasing
gpu_memory_utilization or decreasing max_model_len.
```
Решение: уменьшить `--max-model-len` или увеличить `--gpu-memory-utilization` (но не выше 0.95).

***

## OpenAI-compatible `/v1/chat/completions` API

vLLM реализует тот же HTTP API, что и OpenAI. Это означает, что любой клиент, написанный под `openai` Python SDK, работает с vLLM без изменений — нужно лишь поменять `base_url`.

### Структура запроса

```
POST http://localhost:8000/v1/chat/completions
Content-Type: application/json
Authorization: Bearer EMPTY      ← vLLM принимает любой ключ
```

Тело запроса:
```json
{
  "model": "qwen3-vl",
  "messages": [
    {
      "role": "user",
      "content": [
        {"type": "text", "text": "Что изображено на картинке?"},
        {"type": "image_url", "image_url": {"url": "https://..."}}
      ]
    }
  ],
  "max_tokens": 512,
  "temperature": 0.7,
  "stream": false
}
```

Для чисто текстовых запросов `content` может быть просто строкой: `"content": "Привет!"`.

Изображение можно передать двумя способами:
- **URL**: `{"type": "image_url", "image_url": {"url": "https://example.com/img.jpg"}}`
- **Base64**: `{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,<base64-строка>"}}`

### Синхронный режим (blocking)

```python
from openai import OpenAI
import base64, httpx

client = OpenAI(
    api_key="EMPTY",          # vLLM принимает любой ключ
    base_url="http://localhost:8000/v1",
)

# Загружаем изображение в base64
with httpx.Client() as http:
    img_bytes = http.get("https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/640px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg").content
img_b64 = base64.b64encode(img_bytes).decode()

response = client.chat.completions.create(
    model="qwen3-vl",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Опиши подробно, что на изображении."},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"},
                },
            ],
        }
    ],
    max_tokens=512,
    temperature=0.0,
)

print(response.choices[0].message.content)
print(f"\nИспользовано токенов: {response.usage.total_tokens}")
```

В синхронном режиме `create()` блокирует поток до получения **полного** ответа. Это просто, но для длинных генераций (сотни токенов) пользователь видит задержку перед первым символом.

### Стриминг (Server-Sent Events)

**Как стриминг работает с точки зрения клиента:**

vLLM отдаёт ответ в формате **Server-Sent Events (SSE)** — текстовый поток, где каждая строка выглядит так:

```
data: {"id":"chatcmpl-...","choices":[{"delta":{"content":"Привет"},...}],...}

data: {"id":"chatcmpl-...","choices":[{"delta":{"content":","},...}],...}

data: [DONE]
```

Клиент получает чанки по мере их генерации: каждый чанк содержит 1–несколько токенов. Поле `delta.content` — текст, добавляемый к ответу на этом шаге. Финальный маркер `[DONE]` сигнализирует конец потока. Пользователь видит текст «по буквам» — как в ChatGPT — вместо ожидания полного ответа.

```python
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
)

stream = client.chat.completions.create(
    model="qwen3-vl",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Расскажи про boardwalk на фото."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/640px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
                    },
                },
            ],
        }
    ],
    max_tokens=512,
    temperature=0.7,
    stream=True,   # включаем стриминг
)

print("Ответ модели: ", end="", flush=True)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end="", flush=True)
print()  # перевод строки после конца стрима
```

Несколько тонкостей:
- `delta.content` может быть `None` в первых чанках (пока модель «думает»), поэтому проверяем `if delta.content`
- `flush=True` обязателен при выводе в терминал — иначе Python буферизует вывод и текст появится «пачками»
- Финальный чанк с `finish_reason="stop"` содержит `delta.content = None`; `usage` (если включён) приходит в отдельном завершающем чанке

### Несколько изображений и видео

API поддерживает несколько `image_url` блоков в одном сообщении (до лимита `--limit-mm-per-prompt`):

```python
response = client.chat.completions.create(
    model="qwen3-vl",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Чем отличаются эти два изображения?"},
            {"type": "image_url", "image_url": {"url": "https://...image1.jpg"}},
            {"type": "image_url", "image_url": {"url": "https://...image2.jpg"}},
        ],
    }],
    max_tokens=256,
)
```

Для видео (если модель поддерживает, например Qwen3-VL):

```python
response = client.chat.completions.create(
    model="qwen3-vl",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Что происходит в этом видео?"},
            {"type": "video_url", "video_url": {"url": "https://...video.mp4"}},
        ],
    }],
    max_tokens=256,
)
```

### Проверка доступных моделей

```python
models = client.models.list()
for m in models.data:
    print(m.id)  # выведет "qwen3-vl"
```

Или через curl:
```bash
curl http://localhost:8000/v1/models \
  -H "Authorization: Bearer EMPTY" | python -m json.tool
```